# ACDC initial exploration and basic cleaning
Inspect the original data, identify issues and apply basic corrections.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)

In [ ]:
df_raw = pd.read_csv("../data/raw/acdc_dataset.csv", sep=";")
df = df_raw.copy()
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## Missing values and duplicates

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
pd.Series({
    "Exact duplicate rows": df.duplicated().sum(),
    "Repeated member IDs": df["member_id"].duplicated().sum()
})

## Categories

In [ ]:
df[["gender", "country", "industry", "plan_tier"]].nunique()

In [ ]:
df["plan_tier"].value_counts(dropna=False)

In [ ]:
df["gender"].value_counts(dropna=False)

In [ ]:
df["country"].value_counts(dropna=False).head(15)

In [ ]:
df["industry"].value_counts(dropna=False).head(15)

## Values to review
The age limits below flag records for review. They are not removal rules.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
negatives = (df[numeric_columns] < 0).sum()
negatives[negatives > 0]

In [ ]:
pd.Series({
    "Age below 18 or above 100": ((df["age"] < 18) | (df["age"] > 100)).sum(),
    "Plan with zero days per week": (df["plan_days_per_week"] == 0).sum(),
    "Plan with more than seven days per week": (df["plan_days_per_week"] > 7).sum(),
    "Zero entitled visits": (df["entitled_visits_last_period"] == 0).sum(),
    "Actual visits above entitlement": (df["actual_visits_last_period"] > df["entitled_visits_last_period"]).sum()
})

## Revenue format
A failed conversion does not mean that the original value is missing.

In [ ]:
revenue_numeric = pd.to_numeric(df["lifetime_value"], errors="coerce")
conversion_issues = df["lifetime_value"].notna() & revenue_numeric.isna()

print("Nonmissing values that need format conversion:", conversion_issues.sum())
print("Negative revenue values already readable:", (revenue_numeric < 0).sum())
df.loc[conversion_issues, ["lifetime_value"]].head(8)

## Dates
Parse both observed date formats in a separate table.

In [ ]:
dates = pd.DataFrame(index=df.index)

for column in ["membership_start", "membership_end", "last_badge_swipe"]:
    iso_dates = pd.to_datetime(df[column], format="%Y-%m-%d", errors="coerce")
    other_dates = pd.to_datetime(df[column], format="%d/%m/%Y", errors="coerce")
    dates[column] = iso_dates.fillna(other_dates)

dates.isna().sum()

In [ ]:
dates.agg(["min", "max"])

In [ ]:
end_1970 = dates["membership_end"] == pd.Timestamp("1970-01-01")

pd.Series({
    "End date equal to 1970-01-01": end_1970.sum(),
    "End before start, excluding 1970": ((dates["membership_end"] < dates["membership_start"]) & ~end_1970).sum(),
    "Last swipe before start": (dates["last_badge_swipe"] < dates["membership_start"]).sum(),
    "Last swipe after first end, excluding 1970": ((dates["last_badge_swipe"] > dates["membership_end"]) & ~end_1970).sum()
})

## Initial plots of the original data

In [ ]:
df[["age", "total_visits", "avg_weekly_visits"]].hist(
    bins=30, figsize=(12, 3), layout=(1, 3)
)
plt.tight_layout()
plt.show()

In [ ]:
df["plan_tier"].value_counts().sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("Number of members")
plt.ylabel("Plan before cleaning")
plt.tight_layout()
plt.show()

## Basic cleaning
All changes below affect `df`. The original `df_raw` is preserved.

### Exact duplicates
Remove rows only when every column is identical to avoid counting the same member twice. Then check that member IDs are unique.

In [ ]:
df = df_raw.copy()
rows_before = len(df)
df = df.drop_duplicates().copy()
missing_before_cleaning = df.isna().sum()

print("Rows removed:", rows_before - len(df))
print("Rows remaining:", len(df))
print("Repeated member IDs:", df["member_id"].duplicated().sum())

### Category labels
Remove surrounding spaces, use lowercase and merge clear aliases. This prevents spelling variants from becoming artificial categories. Missing values and other labels are preserved.

In [ ]:
category_columns = ["gender", "country", "industry", "plan_tier"]
categories_before = df[category_columns].nunique()

for column in category_columns:
    df[column] = df[column].str.strip().str.lower()

In [ ]:
df["gender"] = df["gender"].replace({
    "f": "female", "m": "male", "o": "other"
})

df["plan_tier"] = df["plan_tier"].replace({
    "hot desk": "hot_desk", "hotdesk": "hot_desk",
    "full time": "full_time", "fulltime": "full_time",
    "dedicated desk": "dedicated_desk", "dedicateddesk": "dedicated_desk",
    "private office": "private_office", "privateoffice": "private_office"
})

In [ ]:
df["country"] = df["country"].replace({
    "pt": "portugal", "portucal": "portugal",
    "uk": "united kingdom", "gb": "united kingdom",
    "u.k.": "united kingdom",
    "de": "germany", "deutschland": "germany",
    "es": "spain", "espana": "spain", "españa": "spain",
    "fr": "france", "ie": "ireland",
    "nl": "netherlands", "the netherlands": "netherlands",
    "holland": "netherlands"
})

In [ ]:
df["industry"] = df["industry"].replace({
    "it": "information technology",
    "design/creative": "design & creative",
    "design and creative": "design & creative",
    "marketing/pr": "marketing & pr",
    "marketing and pr": "marketing & pr",
    "realestate": "real estate", "real-estate": "real estate"
})

We keep `tech` separate because it is broader than information technology. We also keep `unknown`, `other` and `other (europe)` as distinct labels.

In [ ]:
pd.DataFrame({
    "Categories before": categories_before,
    "Categories after": df[category_columns].nunique()
})

In [ ]:
df["plan_tier"].value_counts(dropna=False)

### Revenue conversion
In this file, commas separate thousands and dots mark decimals. Remove euro symbols, commas and surrounding spaces, then convert to numbers for analysis. These formatting differences were not missing values. Check that conversion creates no new missing values. Negative values remain for later review.

In [ ]:
revenue_text = df["lifetime_value"].str.replace("€", "", regex=False)
revenue_text = revenue_text.str.replace(",", "", regex=False).str.strip()
df["lifetime_value"] = pd.to_numeric(revenue_text, errors="raise")

print("Missing revenue values:", df["lifetime_value"].isna().sum())
print("Negative revenue values:", (df["lifetime_value"] < 0).sum())
print("New missing revenue values:",
      df["lifetime_value"].isna().sum() - missing_before_cleaning["lifetime_value"])

### Date conversion and interpretation
Both date formats were parsed above. Store the parsed dates in the working table.

The value `1970-01-01` looks like a placeholder, but its meaning is not documented. It does not prove that a membership is active. Keep it flagged and avoid using it to calculate duration. The guidelines define `membership_end` as the first end date, so later visits may follow a renewal.

In [ ]:
date_columns = ["membership_start", "membership_end", "last_badge_swipe"]

for column in date_columns:
    df[column] = dates[column]

df[date_columns].dtypes

In [ ]:
end_1970 = df["membership_end"] == pd.Timestamp("1970-01-01")
end_before_start = (df["membership_end"] < df["membership_start"]) & ~end_1970
swipe_before_start = df["last_badge_swipe"] < df["membership_start"]

pd.Series({
    "End date equal to 1970-01-01": end_1970.sum(),
    "End before start, excluding 1970": end_before_start.sum(),
    "Last swipe before start": swipe_before_start.sum()
})

In [ ]:
recency_to_end = (df["membership_end"] - df["last_badge_swipe"]).dt.days
print("Recency values matching membership end:",
      (df["days_since_last_visit"] == recency_to_end).sum())

recency_reference = df["last_badge_swipe"] + pd.to_timedelta(
    df["days_since_last_visit"], unit="D"
)
recency_reference.value_counts().head()

The implied reference date varies across members. Recency often matches the membership end rather than a single extraction date. Its definition needs confirmation before recalculating it.

### Service flags
The original `uses_print` values are only 0 and 1. Convert them to match the other service flags.

In [ ]:
print("Print flag counts before conversion:", df["uses_print"].value_counts().to_dict())
df["uses_print"] = df["uses_print"].astype(bool)
print("Print flag counts after conversion:", df["uses_print"].value_counts().to_dict())
df.filter(like="uses_").dtypes

### Check the basic conversions
At this stage only exact duplicate rows were removed. The format conversions should preserve missing values and all remaining records. Further treatments follow in the next section.

In [ ]:
pd.Series({
    "Rows remaining": len(df),
    "Exact duplicate rows": df.duplicated().sum(),
    "Repeated member IDs": df["member_id"].duplicated().sum(),
    "Missing counts unchanged after conversions": df.isna().sum().equals(missing_before_cleaning)
})

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
negatives = (df[numeric_columns] < 0).sum()
negatives[negatives > 0]

## Agreed data quality treatments
Keep all remaining members. Store the basic cleaned data before applying the agreed rules.

In [ ]:
df_basic = df.copy()
guest_missing_no_use = df["guest_passes_issued"].isna() & ~df["uses_guest_passes"]
meeting_missing_no_use = df["meeting_room_hours"].isna() & ~df["uses_meeting_rooms"]

### Age
Keep age 14. Our project rule treats ages below 10 or above 120 as unreliable for professional membership records. These are explicit assumptions, not universal age limits. Fill missing and invalid ages with the median of the remaining ages. The median is a simple estimate less sensitive to extremes than the mean. Imputation reduces variation and does not recover the true ages.

In [ ]:
age_invalid = (df["age"] < 10) | (df["age"] > 120)
df["age"] = df["age"].mask(age_invalid)
age_imputed = df["age"].isna()
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

print("Invalid ages:", age_invalid.sum())
print("Originally missing ages after deduplication:", df_basic["age"].isna().sum())
print("Median age:", age_median)
print("Ages filled:", age_imputed.sum())
print("Missing ages after treatment:", df["age"].isna().sum())
print("Members aged 14 retained:", (df["age"] == 14).sum())

### Unknown categories
Use `unknown` for missing gender, country and plan labels instead of assigning an unsupported category. Keep these members and their other information. Unknown is distinct from other. These fields can describe clusters later; this does not decide which features will enter the clustering model.

In [ ]:
description_columns = ["gender", "country", "plan_tier"]

for column in description_columns:
    df[column] = df[column].fillna("unknown")

pd.DataFrame({
    "Missing before": df_basic[description_columns].isna().sum(),
    "Unknown after": (df[description_columns] == "unknown").sum(),
    "Missing after": df[description_columns].isna().sum()
})

### Invalid counts and hours
Counts of people, visits, passes, renewals and hours of use cannot represent negative quantities here. Mark these cells as missing while keeping the members. A plan cannot provide more than seven days per week, so mark those cells as missing too. Negative revenue and recency need separate interpretation and remain unchanged.

In [ ]:
negative_columns = [
    "n_colleagues_on_account", "total_visits", "meeting_room_hours",
    "event_space_hours", "guest_passes_issued", "n_renewals"
]
negative_counts_before = (df[negative_columns] < 0).sum()

for column in negative_columns:
    df[column] = df[column].mask(df[column] < 0)

invalid_plan_days = df["plan_days_per_week"] > 7
df["plan_days_per_week"] = df["plan_days_per_week"].mask(invalid_plan_days)

print("Invalid weekly plan days marked missing:", invalid_plan_days.sum())
print("Missing weekly plan days before:", df_basic["plan_days_per_week"].isna().sum())
print("Missing weekly plan days after:", df["plan_days_per_week"].isna().sum())
pd.DataFrame({
    "Negatives before": negative_counts_before,
    "Negatives after": (df[negative_columns] < 0).sum(),
    "Missing after this step": df[negative_columns].isna().sum()
})

### Unknown membership end dates
Every end date equal to 1970-01-01 precedes its recorded membership start. It is not a reliable end date, so store it as NaT, a missing date. The original value remains in `df_raw` and `df_basic`. This does not tell us whether the member is active or inactive.

In [ ]:
known_ends_before = df["membership_end"].notna().sum()
print("All 1970 ends precede membership start:",
      (df.loc[end_1970, "membership_end"] < df.loc[end_1970, "membership_start"]).all())
df["membership_end"] = df["membership_end"].mask(end_1970)

pd.Series({
    "End dates marked unknown": end_1970.sum(),
    "Known ends before": known_ends_before,
    "Known ends after": df["membership_end"].notna().sum(),
    "Missing ends after": df["membership_end"].isna().sum()
})

### Missing service quantities
Guest passes are passes issued by a member for visitors. Assume zero when the original quantity is missing and the usage flag is False. This is an imputation assumption, especially for meeting rooms where some flags conflict with recorded hours.

Keep missing quantities for True flags as NaN. True means the member used the service; it does not tell us the number of passes or hours. Invalid negative quantities remain missing. Preserve recorded positive hours and the original flags in the conflicting records.

In [ ]:
df.loc[guest_missing_no_use, "guest_passes_issued"] = 0
df.loc[meeting_missing_no_use, "meeting_room_hours"] = 0

print("Missing guest pass quantities filled with zero:", guest_missing_no_use.sum())
print("Missing meeting hours filled with zero:", meeting_missing_no_use.sum())
service_checks = []
for quantity, flag in [("guest_passes_issued", "uses_guest_passes"),
                       ("meeting_room_hours", "uses_meeting_rooms")]:
    for used in [False, True]:
        before = df_basic.loc[df_basic[flag] == used, quantity]
        after = df.loc[df[flag] == used, quantity]
        service_checks.append({
            "Quantity": quantity, "Used": used,
            "Missing before": before.isna().sum(),
            "Missing after": after.isna().sum(),
            "Positive before": (before > 0).sum(),
            "Positive after": (after > 0).sum()
        })
pd.DataFrame(service_checks)

### Unresolved data quality flags
The flags below refer to values before these treatments. No extra correction is made to visit discrepancies; the rule for negative values still applies. Their cause is unknown. Date conflicts remain unresolved, and future duration features must exclude affected intervals.

A zero revenue value together with recency 999 suggests a possible data quality pattern. It does not establish that the member is inactive or that the whole record is false. Keep both values unchanged. A cluster driven by these values could reflect data quality rather than a useful business segment. Compare results with and without this flagged group during modelling before deciding whether to exclude it.

In [ ]:
visit_gap = df_basic["actual_visits_last_period"] - df_basic["total_visits"]
meeting_conflict = (~df_basic["uses_meeting_rooms"]) & (df_basic["meeting_room_hours"] > 0)

quality_flags = pd.DataFrame({
    "First end date unresolved (1970)": end_1970,
    "First end before start": end_before_start,
    "Last swipe before start": swipe_before_start,
    "Visits exceed total by one or two": visit_gap.between(1, 2),
    "Visits exceed total by more than two": visit_gap > 2,
    "Positive meeting hours with unused flag": meeting_conflict,
    "Zero revenue and recency 999": (df_basic["lifetime_value"] == 0) & (df_basic["days_since_last_visit"] == 999)
})
quality_flags.sum()

In [ ]:
zero_999 = quality_flags["Zero revenue and recency 999"]
pd.Series({
    "Members flagged": zero_999.sum(),
    "Share of members (%)": round(100 * zero_999.mean(), 2),
    "Flagged members with positive visits": (zero_999 & (df_basic["total_visits"] > 0)).sum(),
    "Flagged members with positive renewals": (zero_999 & (df_basic["n_renewals"] > 0)).sum(),
    "Members retained": len(df)
})

In [ ]:
valid_visit_totals = df_basic["total_visits"] >= 0
visit_columns = ["total_visits", "actual_visits_last_period"]
pd.Series({
    "Positive meeting hours in conflicts preserved": df.loc[meeting_conflict, "meeting_room_hours"].equals(df_basic.loc[meeting_conflict, "meeting_room_hours"]),
    "Original service flags preserved": df.filter(like="uses_").equals(df_basic.filter(like="uses_")),
    "Nonnegative visit counts preserved": (df.loc[valid_visit_totals, visit_columns] == df_basic.loc[valid_visit_totals, visit_columns]).all().all(),
    "Revenue and recency preserved": df[["lifetime_value", "days_since_last_visit"]].equals(df_basic[["lifetime_value", "days_since_last_visit"]]),
    "Start and swipe dates preserved": df[["membership_start", "last_badge_swipe"]].equals(df_basic[["membership_start", "last_badge_swipe"]]),
    "Other end dates preserved": df.loc[~end_1970, "membership_end"].equals(df_basic.loc[~end_1970, "membership_end"])
})

### Check the treatments
Confirm that all members remain and compare missing values before and after the agreed treatments. Missing values can increase when unreliable measurements or dates are identified. Original values remain available in the saved copies. Quality flags and imputation masks are kept separately from the dataset.

In [ ]:
print("Members before treatments:", len(df_basic))
print("Members after treatments:", len(df))
missing_comparison = pd.DataFrame({
    "Before agreed treatments": df_basic.isna().sum(),
    "After agreed treatments": df.isna().sum()
})
missing_comparison[(missing_comparison > 0).any(axis=1)]

## Distributions, relationships and recorded trends
Explore usage, economic value, plans and reported service use. These analyses use separate views and summaries. All members remain in `df`, including the flagged zero revenue and recency 999 group. No clustering or final model preparation is performed here.

### Numeric distributions
Compare counts, missing values, zeros, medians and upper percentiles. The full maxima remain visible here even when a later chart uses a closer view. Different metrics have different numbers of available observations.

In [ ]:
summary_metrics = [
    "avg_weekly_visits", "total_visits", "actual_visits_last_period",
    "meeting_room_hours", "event_space_hours", "guest_passes_issued",
    "n_renewals", "lifetime_value"
]
metric_summary = df[summary_metrics].describe(percentiles=[0.5, 0.95, 0.99]).T
metric_summary["Missing"] = df[summary_metrics].isna().sum()
metric_summary["Zeros"] = (df[summary_metrics] == 0).sum()
metric_summary[["count", "Missing", "Zeros", "min", "mean", "50%", "95%", "99%", "max"]].round(2)

### Economic value and extreme values
For economic comparisons, use observed nonnegative lifetime values. Negative values need interpretation and are excluded only from this analytical view. Keep all zeros, including the flagged group. This does not change revenue in `df`.

In [ ]:
revenue_data = df.loc[df["lifetime_value"].notna() & (df["lifetime_value"] >= 0)].copy()
pd.Series({
    "All members": len(df),
    "Observed nonnegative revenue": len(revenue_data),
    "Missing revenue excluded from this view": df["lifetime_value"].isna().sum(),
    "Negative revenue excluded from this view": (df["lifetime_value"] < 0).sum(),
    "Zero revenue retained": (revenue_data["lifetime_value"] == 0).sum()
})

In [ ]:
revenue_limit = revenue_data["lifetime_value"].quantile(0.99)
revenue_zoom = revenue_data.loc[revenue_data["lifetime_value"] <= revenue_limit]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
revenue_data["lifetime_value"].plot.hist(bins=40, ax=axes[0])
axes[0].set_title(f"Full observed range, n = {len(revenue_data):,}")
revenue_zoom["lifetime_value"].plot.hist(bins=30, ax=axes[1])
axes[1].set_title(f"View up to P99, n = {len(revenue_zoom):,}")
for ax in axes:
    ax.set_xlabel("Lifetime value")
    ax.set_ylabel("Members")
plt.tight_layout()
plt.show()
print("Upper limit of the closer view:", round(revenue_limit, 2))
print("Members outside the closer view, still retained:", len(revenue_data) - len(revenue_zoom))

Lifetime value is strongly skewed: a few very large values stretch the full range. The closer view shows the distribution for most members without deleting the upper tail. High values may include valuable customers as well as data problems, so this chart alone does not justify removal.

### Usage and value by plan
Compare medians because means are sensitive to extreme values. Show group sizes and revenue coverage. Usage medians use available observations; revenue medians use the analytical view above. Unknown remains a separate descriptive group.

In [ ]:
plan_order = ["hot_desk", "flex", "dedicated_desk", "full_time", "private_office", "unknown"]
plan_summary = df.groupby("plan_tier")[["avg_weekly_visits", "actual_visits_last_period"]].median()
plan_summary["Members"] = df.groupby("plan_tier").size()
plan_summary["Revenue observations"] = revenue_data.groupby("plan_tier")["lifetime_value"].count()
plan_summary["Median lifetime value"] = revenue_data.groupby("plan_tier")["lifetime_value"].median()
plan_summary["Revenue missing (%)"] = df["lifetime_value"].isna().groupby(df["plan_tier"]).mean() * 100
plan_summary = plan_summary.reindex(plan_order)
plan_summary.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plan_summary["avg_weekly_visits"].plot.barh(ax=axes[0])
axes[0].set_title("Median weekly visits by current plan")
axes[0].set_xlabel("Visits per week")
plan_summary["Median lifetime value"].plot.barh(ax=axes[1])
axes[1].set_title("Median observed nonnegative lifetime value")
axes[1].set_xlabel("Lifetime value")
axes[0].set_ylabel("Current plan")
plt.tight_layout()
plt.show()

Private office members have the highest observed median lifetime value, while dedicated desk members have higher median weekly attendance. Value and usage therefore describe different aspects of customers. These are associations, not effects caused by the plan. Lifetime value covers potentially different membership lengths, and the current plan may not describe the entire history.

### Relationship between visits and value
Use members with both nonnegative revenue and known total visits. Compare the full scatter with a closer view of the same pairs. Axis limits only change the display. They do not remove members.

In [ ]:
visit_value_pairs = revenue_data[["total_visits", "lifetime_value"]].dropna()
visits_limit = visit_value_pairs["total_visits"].quantile(0.99)
value_limit = visit_value_pairs["lifetime_value"].quantile(0.99)
inside_view = (visit_value_pairs["total_visits"] <= visits_limit) & (visit_value_pairs["lifetime_value"] <= value_limit)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax in axes:
    ax.scatter(visit_value_pairs["total_visits"], visit_value_pairs["lifetime_value"], alpha=0.2, s=8)
    ax.set_xlabel("Total visits")
    ax.set_ylabel("Lifetime value")
axes[0].set_title(f"Full range, n = {len(visit_value_pairs):,}")
axes[1].set_xlim(0, visits_limit)
axes[1].set_ylim(0, value_limit)
axes[1].set_title(f"P99 axis limits, visible n = {inside_view.sum():,}")
plt.tight_layout()
plt.show()
print("Members without an analytical pair:", len(df) - len(visit_value_pairs))
print("Pairs outside the closer view:", (~inside_view).sum())
print("Full sample Pearson correlation:", round(visit_value_pairs.corr().iloc[0, 1], 3))

The closer scatter makes the association easier to see, while the full view reveals the extremes. Pearson correlation describes a linear relationship and is sensitive to extreme values. Compare it with the plots. The matrix below uses available pairs, so sample sizes differ. It does not demonstrate causation.

In [ ]:
correlation_columns = [
    "total_visits", "avg_weekly_visits", "actual_visits_last_period",
    "meeting_room_hours", "n_renewals", "lifetime_value"
]
correlation_data = df[correlation_columns].copy()
correlation_data["lifetime_value"] = correlation_data["lifetime_value"].mask(correlation_data["lifetime_value"] < 0)
correlation_data.corr().round(2)

### Individual and team accounts
Compare group sizes and medians, then service use percentages. Percentages avoid confusing the larger individual group with higher propensity to use an extra. Service flags remain unchanged, including the recorded meeting room conflicts.

In [ ]:
account_summary = df.groupby("is_team_account")[["avg_weekly_visits", "actual_visits_last_period"]].median()
account_summary["Members"] = df.groupby("is_team_account").size()
account_summary["Revenue observations"] = revenue_data.groupby("is_team_account")["lifetime_value"].count()
account_summary["Median lifetime value"] = revenue_data.groupby("is_team_account")["lifetime_value"].median()
account_summary.rename(index={False: "Individual", True: "Team"}).round(2)

In [ ]:
service_columns = df.filter(like="uses_").columns
service_rates = df.groupby("is_team_account")[service_columns].mean().T * 100
service_rates = service_rates.rename(columns={False: "Individual", True: "Team"})
service_rates.index = service_rates.index.str.replace("uses_", "", regex=False).str.replace("_", " ", regex=False)
service_rates.plot.barh(figsize=(9, 5))
plt.xlabel("Members with service flag True (%)")
plt.ylabel("Service")
plt.title("Reported service use: individual and team accounts")
plt.legend(title="Account type")
plt.tight_layout()
plt.show()
print("Account sample sizes:", account_summary["Members"].rename(index={False: "Individual", True: "Team"}).to_dict())
service_rates.round(2)

The two account types have similar median attendance and reported service use percentages. Account type alone does not show a strong descriptive separation here. Rare services such as bike lockers should not be interpreted as broad customer patterns. The flags describe whether an extra was used, not its intensity.

### Recorded membership starts over time
Count the first membership starts represented in the dataset. This is not a verified time series of acquisitions, revenue or current activity. Dates have not been reconstructed from unresolved end dates.

In [ ]:
monthly_starts = df.groupby(df["membership_start"].dt.to_period("M")).size()
monthly_starts.index = monthly_starts.index.to_timestamp()
monthly_starts.plot(figsize=(11, 3), marker="o", markersize=3)
plt.xlabel("Recorded first membership start")
plt.ylabel("Members")
plt.title(f"Recorded membership starts by month, n = {monthly_starts.sum():,}")
plt.tight_layout()
plt.show()
print("Recorded starts on 2021-01-01:", (df["membership_start"] == pd.Timestamp("2021-01-01")).sum())
df.groupby(df["membership_start"].dt.year).size().rename("Members")

Recorded starts are concentrated in 2021, with a separate increase in late 2023. The unusually large count on the first day of 2021 may reflect a recording or observation boundary effect. Its cause is unknown, so the dates are preserved. These counts do not establish business growth or decline.

### Sensitivity to the flagged revenue pattern
Compare value summaries with and without the zero revenue and recency 999 group in a temporary view. This evaluates how the flagged data pattern affects the EDA. It does not remove the group or validate an exclusion rule.

In [ ]:
revenue_without_flagged = revenue_data.loc[~zero_999]
value_sensitivity = pd.DataFrame({
    "Flagged group included": revenue_data["lifetime_value"].agg(["count", "median", "mean"]),
    "Flagged group omitted only from this view": revenue_without_flagged["lifetime_value"].agg(["count", "median", "mean"])
})
value_sensitivity.round(2)

### Main findings from these comparisons
Value and attendance differ across plans, but do not follow the same ordering. Individual and team accounts show similar reported service use. Extreme values and the flagged revenue pattern materially affect numeric summaries. These findings motivate careful feature selection and business interpretation; they are not discovered clusters.

## Next steps
Develop and justify business features, then formulate segmentation hypotheses and prepare the discussion slides. Keep unresolved quality decisions explicit. Final clustering, scaling and algorithm comparison belong to the later deliverable.